In [1]:
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer
from transformers import TFBertForSequenceClassification
import numpy as np
import re
import json

2024-12-02 23:37:29.352597: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-12-02 23:37:29.367635: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-12-02 23:37:29.372143: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-12-02 23:37:29.383648: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-12-02 23:37:30.294989: W tensorflow/compiler/tf2

In [2]:
# Load the datasets
# Setting up the first dataset
df_phishing = pd.read_csv('urlData/phishing_site_urls.csv')
df_phishing.rename(columns={'URL': 'Url'}, inplace=True)
df_phishing['Label'] = df_phishing['Label'].apply(lambda label: 1 if label == 'bad' else 0)

# Setting up the second dataset
df_urlset = pd.read_csv('urlData/urlset.csv', encoding='ISO-8859-1', on_bad_lines='skip')
df_urlset.rename(columns={'domain': 'Url', 'label': 'Label'}, inplace=True)
df_urlset.dropna(inplace=True)
df_urlset['Label'] = df_urlset['Label'].astype(int)
df_urlset = df_urlset[['Url', 'Label']]

# Load the JSON file
with open('urlData/urls.json', 'r') as f:
    json_data = json.load(f)

# Convert JSON data to DataFrame
df_json = pd.DataFrame(json_data)
df_json.rename(columns={'text': 'Url', 'label': 'Label'}, inplace=True)

# # Combine the datasets
df = pd.concat([df_phishing, df_urlset, df_json], ignore_index=True)
# #df = pd.concat([df_phishing], ignore_index=True)
df = df[['Url','Label']]


# # Remove duplicates from the combined dataset
df.drop_duplicates(subset='Url', inplace=True)
df.dropna(inplace=True)

# # Preprocess the data
X = df['Url'].tolist()  # All URLs
y = df['Label'].tolist()  # 1 for phishing, 0 for legitimate

#temp tests
# X = X[:100]
# y = y[:100]

/tmp/ipykernel_418/193132322.py:8: DtypeWarning: Columns (1,2,3,11,12) have mixed types. Specify dtype option on import or set low_memory=False.
  df_urlset = pd.read_csv('urlData/urlset.csv', encoding='ISO-8859-1', on_bad_lines='skip')


In [3]:
#Clearning the url
def clean_url(url):
    # Replace 'https://' first, then 'http://' and 'www.'
    url = re.sub(r'^https?://', '', url)  # This will remove both 'http://' and 'https://'
    url = re.sub(r'^www\.', '', url)      # This will remove 'www.'
    if url.endswith('/'):
        url = url[:-1]
    return url
X = [clean_url(url) for url in X]

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# # Load the dataset
# df = pd.read_csv('phishing_site_urls.csv')

# # Preprocess the data
# X = df['URL'].tolist()
# y = (df['Label'] == 'bad').astype(int).tolist()

# Initialize the BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Tokenize and encode the URLs
train_encodings = tokenizer(X_train, truncation=True, padding=True, max_length=64, return_tensors='tf')
test_encodings = tokenizer(X_test, truncation=True, padding=True, max_length=64, return_tensors='tf')

# Convert to TensorFlow datasets with attention masks
train_dataset = tf.data.Dataset.from_tensor_slices((
    {
        'input_ids': train_encodings['input_ids'],
        'attention_mask': train_encodings['attention_mask']
    },
    tf.constant(y_train)
)).shuffle(1000).batch(16)

test_dataset = tf.data.Dataset.from_tensor_slices((
    {
        'input_ids': test_encodings['input_ids'],
        'attention_mask': test_encodings['attention_mask']
    },
    tf.constant(y_test)
)).batch(16)

# Initialize the BERT model for sequence classification
model = TFBertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

# Compile the model
optimizer = tf.keras.optimizers.Adam(learning_rate=2e-5)
loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
metric = tf.keras.metrics.SparseCategoricalAccuracy('accuracy')
model.compile(optimizer=optimizer, loss=loss, metrics=[metric])

# Train the model
history = model.fit(train_dataset, epochs=1, validation_data=test_dataset)

# Evaluate the model
test_loss, test_accuracy = model.evaluate(test_dataset)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

# Save the model
model.save_weights('phishing_classifier_model_tf')

# Function to predict if a URL is phishing or not
def predict_url(url):
    inputs = tokenizer(url, truncation=True, padding=True, max_length=64, return_tensors='tf')
    outputs = model({'input_ids': inputs['input_ids'], 'attention_mask': inputs['attention_mask']})
    prediction = tf.nn.softmax(outputs.logits, axis=-1)
    label = tf.argmax(prediction, axis=-1).numpy()[0]
    return "Phishing" if label == 1 else "Not Phishing"

# Example usage
test_url = "https://example.com"
result = predict_url(test_url)
print(f"The URL '{test_url}' is predicted to be: {result}")

In [ ]:
testur = "googlehacking.ca"
res = predict_url(testur)
res

In [ ]:
list = []

for i in range(len(y_train)):
    if y_train [i] == 1:
        list.append(X_train[i])
    

In [ ]:
list

In [ ]:
df_phishing = 0
df_urlset = 0
df_json = 0
train_encodings = 0
test_encodings = 0
train_dataset = 0
test_dataset = 0
list = 0

In [ ]:
import tf2onnx

# Define the model input signature for conversion
spec = (tf.TensorSpec((None, 64), tf.int32, name="input_ids"), 
        tf.TensorSpec((None, 64), tf.int32, name="attention_mask"))

# Convert the TensorFlow model to ONNX format
output_path = "model64tk2.onnx"
model_proto, _ = tf2onnx.convert.from_keras(
    model,
    input_signature=spec,
    opset=13,  # Choose a suitable ONNX opset version (13 is widely compatible)
    output_path=output_path
)

print(f"ONNX model saved to {output_path}")

# Verify the ONNX model
import onnx

onnx_model = onnx.load(output_path)
onnx.checker.check_model(onnx_model)

print("ONNX model is valid.")